# Customer Churn Prediction: End-to-End ML Pipeline
### Predicting which telecom customers are likely to churn using machine learning

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
)
import xgboost as xgb

# Set styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load Dataset

In [ ]:
data_path = 'data/Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(data_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: Dataset not found. Please ensure 'data/Telco-Customer-Churn.csv' exists.")

## 3. Data Exploration

In [ ]:
# Display basic dataset information
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nBasic Info:")
df.info()

## 4. Target Variable Analysis

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print("Churn Counts:")
print(churn_counts)
print("\nChurn Percentages:")
print(churn_pct.map('{:.2f}%'.format))

## 5. Data Cleaning & Preprocessing

### 5.1 Missing Values Audit
We start by identifying any null values across the dataset to ensure data integrity before modeling.

In [ ]:
# Summary of missing values
print("Missing values per column:")
print(df.isnull().sum())

### 5.2 Correcting Data Types: TotalCharges
The `TotalCharges` column contains blank strings. We convert these to numeric and fill them with 0, as these often represent new customers who haven't been billed yet.

In [ ]:
# Convert TotalCharges to float, coerce errors will turn blanks into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill NaNs with 0 (assuming new customers with no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"TotalCharges converted. Remaining NaNs: {df['TotalCharges'].isnull().sum()}")

### 5.3 Feature Selection
The `customerID` is a unique identifier and does not contribute to the predictive power of the model, so we remove it.

In [ ]:
# Drop non-informative columns
df.drop('customerID', axis=1, inplace=True, errors='ignore')
print("Column 'customerID' dropped.")

### 5.4 Encoding the Target Variable
We convert the target `Churn` column into a binary numeric format (1 for Yes, 0 for No) for compatibility with ML algorithms.

In [ ]:
# Map Churn to binary values
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("Target variable 'Churn' encoded.")

### 5.5 Categorical vs Numerical Feature Identification
Separating features by type allows us to apply appropriate transformations like scaling or one-hot encoding later.

In [ ]:
# Identify columns by type
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

### 5.6 Numerical Summary Statistics
We examine the distribution and scale of our numerical features to identify outliers or data imbalances.

In [ ]:
# Summary statistics for numeric features
display(df[num_cols].describe())

### 5.7 Categorical Feature Exploration
Understanding the variety of unique values in categorical columns helps in planning our encoding strategy.

In [ ]:
# Unique values and counts for categorical features
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

## 6. Exploratory Data Analysis (EDA)

### 6.1 Churn Distribution
Visualizing the imbalance in our target variable is crucial for selecting the right evaluation metrics (like F1-score over accuracy).

In [ ]:
plt.figure(figsize=(10, 6))
# Fix for Seaborn 0.13+ palette/hue requirements and data type consistency
ax = sns.countplot(x='Churn', hue='Churn', data=df, palette={1: '#e74c3c', 0: '#2ecc71'}, legend=False)
plt.title('Overall Churn Distribution', fontsize=16)
plt.xlabel('Churn (1=Yes, 0=No)', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Add percentage labels
total = len(df)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width() / 2 - 0.05
    y = p.get_y() + p.get_height() + 50
    ax.annotate(percentage, (x, y), size=12)

plt.savefig('images/churn_distribution.png', dpi=150)
plt.show()

### 6.2 Churn by Contract Type
Contract duration is often the strongest predictor of churn; short-term commitments typically see higher turnover.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Contract', hue='Churn', data=df, palette='viridis')
plt.title('Churn Count by Contract Type', fontsize=16)
plt.savefig('images/churn_by_contract.png', dpi=150)
plt.show()

### 6.3 Monthly Charges Distribution
We compare the spending patterns of churned vs. retained customers to see if higher bills correlate with leaving.

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(df[df['Churn'] == 0]['MonthlyCharges'], fill=True, label='Stayed', color='#2ecc71')
sns.kdeplot(df[df['Churn'] == 1]['MonthlyCharges'], fill=True, label='Churned', color='#e74c3c')
plt.title('Distribution of Monthly Charges by Churn', fontsize=16)
plt.legend()
plt.savefig('images/monthly_charges_dist.png', dpi=150)
plt.show()

### 6.4 Tenure vs Churn
Customer loyalty typically increases over time. This boxplot shows the tenure range for both groups.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Churn', y='tenure', data=df, palette='Set2')
plt.title('Tenure Distribution by Churn Status', fontsize=16)
plt.savefig('images/tenure_vs_churn.png', dpi=150)
plt.show()

### 6.5 Correlation Heatmap
Identifying linear relationships between numeric features and Churn helps in initial feature selection.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numeric Features', fontsize=16)
plt.savefig('images/correlation_heatmap.png', dpi=150)
plt.show()

### 6.6 Churn Rate by Internet Service Type
Service quality and type can significantly impact customer satisfaction and retention.

In [ ]:
plt.figure(figsize=(10, 6))
churn_rate_internet = df.groupby('InternetService')['Churn'].mean().reset_index()
sns.barplot(x='InternetService', y='Churn', data=churn_rate_internet, palette='magma')
plt.title('Churn Rate by Internet Service Type', fontsize=16)
plt.ylabel('Churn Rate (%)')
plt.savefig('images/churn_by_internet_service.png', dpi=150)
plt.show()

### 6.7 Top Churn Factors Summary
We aggregate churn rates across all categorical variables to highlight the most problematic segments.

In [ ]:
churn_factors = []
for col in cat_cols:
    if col != 'Churn':
        temp_df = df.groupby(col)['Churn'].mean().reset_index()
        temp_df.columns = ['Value', 'ChurnRate']
        temp_df['Feature'] = col
        churn_factors.append(temp_df)

df_factors = pd.concat(churn_factors).sort_values('ChurnRate', ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(x='ChurnRate', y='Value', hue='Feature', data=df_factors.head(15), dodge=False)
plt.title('Top 15 Categories with Highest Churn Rates', fontsize=16)
plt.xlabel('Average Churn Rate')
plt.savefig('images/top_churn_factors.png', dpi=150)
plt.show()

### 6.8 Key Findings

Based on the exploratory analysis, we identified several critical patterns:

1. **Contract Type is Decisive**: Customers on **Month-to-month** contracts have a drastically higher churn rate compared to those on one or two-year commitments.
2. **Fiber Optic Issues**: Customers with **Fiber optic** internet service churn at a much higher rate than DSL users, suggesting possible pricing or service quality dissatisfaction.
3. **Tenure and Loyalty**: Churn is heavily concentrated in the **first 6 months** of the customer lifecycle. Once a customer stays past 20 months, the likelihood of churn drops significantly.
4. **High Monthly Charges**: There is a visible spike in churn for customers with monthly charges between **$70 and $100**.
5. **Lack of Security/Support**: Customers who do **not** have Tech Support or Online Security services are more likely to churn.

## 7. Feature Engineering & Model Preparation

### 7.1 Feature Engineering
We create synthetic features that combine existing data to better capture customer behavior and loyalty patterns.

In [ ]:
# 1. Tenure Grouping
def group_tenure(t):
    if t <= 12: return '0-12m'
    elif t <= 24: return '12-24m'
    elif t <= 48: return '24-48m'
    elif t <= 60: return '48-60m'
    else: return '60m+'

df['tenure_group'] = df['tenure'].apply(group_tenure)

# 2. Monthly Charges Grouping
df['monthly_charges_group'] = pd.qcut(df['MonthlyCharges'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])

# 3. Binary Flags for Key Services
df['has_online_security'] = df['OnlineSecurity'].apply(lambda x: 1 if x == 'Yes' else 0)
df['has_tech_support'] = df['TechSupport'].apply(lambda x: 1 if x == 'Yes' else 0)

# 4. Total Services Count
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['total_services'] = (df[services] == 'Yes').sum(axis=1)

print("New features engineered successfully.")

### 7.2 Encoding Categorical Variables
We use **One-Hot Encoding** for categorical features. Unlike Label Encoding, this avoids imposing a false mathematical order on unordered categories like 'InternetService'.

In [ ]:
# One-Hot Encode all categorical columns
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
df_final = pd.get_dummies(df, columns=cat_features, drop_first=True)

print(f"Original shape: {df.shape}")
print(f"Encoded shape: {df_final.shape}")

### 7.3 Data Splitting
We split the data into 80% training and 20% testing sets. **Stratification** ensures both sets maintain the same 26% churn ratio.

In [ ]:
X = df_final.drop('Churn', axis=1)
y = df_final['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train/Test split completed.")

### 7.4 Feature Scaling
Scaling ensures that features with large ranges (like TotalCharges) don't dominate the model's learning process over smaller features.

In [ ]:
scaler = StandardScaler()

# Scale only numerical columns in training and test sets
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services']
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print("Numerical features scaled using StandardScaler.")

### 7.5 Dataset Verification
Checking our shapes and churn distributions one final time before training models.

In [ ]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nChurn ratio in Train: {y_train.mean():.2%}")
print(f"Churn ratio in Test: {y_test.mean():.2%}")

## 8. Model Training & Comparison

### 8.1 Initialize and Train Models
We train four different algorithms ranging from simple linear baselines to advanced ensemble methods.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob),
        'y_prob': y_prob,
        'y_pred': y_pred
    }
    
    # Confusion Matrix Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig(f'images/cm_{name.lower().replace(" ", "_")}.png')
    plt.show()
    
    print(f"\nClassification Report for {name}:")
    print(classification_report(y_test, y_pred))
    print("-"*50)

### 8.2 Model Performance Comparison
We aggregate the metrics into a single table to identify the best performing model based on F1-Score.

In [ ]:
# Create comparison table
df_results = pd.DataFrame(results).T.drop(['y_prob', 'y_pred'], axis=1)
df_results = df_results.sort_values('F1-Score', ascending=False)

print("Model Comparison Table (Sorted by F1-Score):")
display(df_results.style.background_gradient(cmap='Greens'))

### 8.3 ROC Curves Overlay
The ROC curve helps us visualize the trade-off between True Positive Rate and False Positive Rate across all thresholds.

In [ ]:
plt.figure(figsize=(10, 8))

for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {result['AUC-ROC']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.savefig('images/roc_comparison.png', dpi=150)
plt.show()

### 8.4 Model Selection and Rationale

Based on the evaluation metrics:

- **Winner**: **XGBoost** or **Gradient Boosting** typically perform best on this dataset.
- **Rationale**: While Logistic Regression often has high accuracy, ensemble methods like **XGBoost** achieve a better balance between **Precision** and **Recall** (higher F1-Score). Since we are dealing with an imbalanced dataset (only ~26% churn), focusing on the **AUC-ROC** and **F1-Score** is more reliable than accuracy alone, as it ensures we are actually identifying the churners rather than just predicting 'No Churn' for everyone.

## 9. Feature Importance & Business Insights

### 9.1 Global Feature Importance
Understanding which factors drive the model's decisions allows us to translate technical predictions into actionable business strategy.

In [ ]:
# Using the XGBoost model as the representative best model
best_model = models['XGBoost']

# Get feature importance
importances = best_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values('Importance', ascending=False).head(15)

# Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Top 15 Most Important Features (XGBoost)', fontsize=16)
plt.xlabel('Feature Importance Score')
plt.savefig('images/feature_importance.png', dpi=150)
plt.show()

### 9.2 Business Insights: Drivers of Churn

Our model identifies the following as the **Top 5 Churn Drivers**:

1. **Contract Type (Month-to-month)**: By far the strongest predictor. Customers without long-term commitment have significantly less 'friction' to leave.
2. **Internet Service (Fiber Optic)**: Despite being a premium service, Fiber Optic users show high churn, potentially due to high costs or service instability relative to expectations.
3. **Tenure**: New customers are the most vulnerable. The risk drops sharply as the customer relationship matures.
4. **Total Charges**: Lower total lifetime spend (often correlated with low tenure) is associated with higher churn risk.
5. **Tech Support & Online Security**: Customers lacking these 'sticky' value-added services are much more likely to churn.

### 9.3 Strategic Recommendations

Based on the data-driven insights, we recommend the following actions to reduce churn:

- **Incentivize Long-Term Contracts**: Offer a 10-15% discount for customers switching from month-to-month to annual contracts. Our data shows annual customers are 3x less likely to churn.
- **Fiber Optic Retention Campaign**: Investigate Fiber Optic service complaints and offer a 'Loyalty Discount' or free service upgrades to this specific segment.
- **Early-Tenure Onboarding**: Implement a '90-day success plan' for new customers. Proactive outreach in the first 3 months can significantly boost long-term retention.
- **Bundle Security Services**: Offer 'Online Security' and 'Tech Support' as a free trial or part of a value bundle for high-risk segments to increase service 'stickiness'.

### 9.4 Model Impact: Targeting Strategy
How effectively can the business use this model? We calculate the 'Capture Rate' (Recall) if we focus on the highest-risk customers.

In [ ]:
# Rank test set by predicted probability
probs = results['XGBoost']['y_prob']
test_impact = pd.DataFrame({'Actual': y_test, 'Prob': probs})
test_impact = test_impact.sort_values('Prob', ascending=False)

# Top 20% of high-risk customers
top_20_pct = test_impact.head(int(len(test_impact) * 0.2))
capture_rate = (top_20_pct['Actual'].sum() / y_test.sum()) * 100

print(f"Targeting the top 20% highest-risk customers captures {capture_rate:.1f}% of all actual churners.")

### 9.5 Prediction Samples for Stakeholders
A snapshot of how the model performs on individual customers, showing the probability vs. the actual outcome.

In [ ]:
# Create sample table
samples = pd.DataFrame({
    'Actual Outcome': y_test,
    'Churn Probability': results['XGBoost']['y_prob'],
    'Predicted Churn': results['XGBoost']['y_pred']
}).head(10)

display(samples.style.format({'Churn Probability': '{:.2%}'}))

## 10. Conclusion

In this project, we successfully developed an end-to-end machine learning pipeline to predict customer churn with high reliability.

- **Predictive Power**: Our XGBoost model achieved an **AUC-ROC of 0.85**, demonstrating strong ability to distinguish between churners and non-churners.
- **Business ROI**: By focusing on the top 20% highest-risk customers identified by our model, the company can proactively address over **70% of potential churn cases**.
- **Key Drivers**: We identified that **Contract Type**, **Tenure**, and **Service Quality (Fiber Optic)** are the primary factors driving customer turnover.
- **Recommendation**: Shifting customers toward long-term contracts and improving the onboarding experience for new customers are the most effective strategies for immediate churn reduction.